# Test io notebook:

This notebook is used to test the implementation of **jklab-core/io.py** module.

Testing (roughly) follows this routine:

0. import the module to test;
1. select component (variable/class/function) to test;
2. enstablish the correct behaviour;
3. implement a local function to assert the results;
4. summarise which tests have been passed.

## Module import

In [1]:
import pytest
import numpy as np

import jklab.core.exceptions as jkex
import jklab.core.io as jkio

## Testing constants

In [2]:
_dir = "./test_data/io"
fake_input_dir = f"{_dir}/input"
fake_output_dir = f"{_dir}/output"

## Testing helpers

In [3]:
# Collection of all test results in this notebook
test_results = {}

# Single test record function
def record_test(
    test_name,
    condition
):
    """
    Record and display the result of a test.
    """

    passed_flag = bool(condition)

    test_results[test_name] = passed_flag

    if passed_flag:
        print(f"✅ PASSED: {test_name}.")
    else:
        print(f"❌ FAILED: {test_name}.")


# Test summary function
def print_test_summary(
    test_results
):
    """
    Print a summary of test results.
    """

    # Used for visual separation
    separator_len = 40

    total = len(test_results)
    passed = sum(test_results.values())
    failed = total - passed

    print()
    print("=" * separator_len)
    print("TEST SUMMARY")
    print("=" * separator_len)

    print(f"Passed: {passed}/{total}")
    print(f"Failed: {failed}/{total}")

    if total:
        ratio = passed / total * 100
        print(f"Success rate: {ratio:.1f}%")

    print()

    for name, result in test_results.items():

        status = "PASSED" if result else "FAILED"

        print(f"{status}: {name}")

    print("=" * separator_len)


def test_raises(
    func,
    expected_error,
    expected_message=None,
):

    try:
        func()

    except expected_error as error:

        if expected_message is None:
            return True

        return expected_message in str(error)

    except Exception:
        return False

    return False

## 

## Test 1 - Data loading

In [4]:
def test_load_txt(
    path,
    dtype=float,
    skiprows=0,
    usecols=None,
    start=None,
    stop=None,
):

    # Load file via numpy
    numpy_data = np.loadtxt(
        fname=path,
        dtype=dtype,
        skiprows=skiprows,
        usecols=usecols
    )[start:stop]

    # Load file via jklab
    jklab_data = jkio.load_txt(
        path=path,
        dtype=dtype,
        skiprows=skiprows,
        usecols=usecols,
    )[start:stop]

    # Compare dtype
    dtype_match = (
        jklab_data.dtype
        == numpy_data.dtype
    )
    # Compare shape
    shape_match = (
        jklab_data.shape
        == numpy_data.shape
    )

    # Compare elements
    data_match = np.allclose(
        jklab_data,
        numpy_data,
    )

    test = (
        data_match
        and dtype_match
        and shape_match
    )

    return test

### Basic loading

In [5]:
# === Input ===
# Pick a file to load
data_path = f"{fake_input_dir}/fake_data_float.dat"

# === Test & Record ===
record_test(
    test_name="load_txt: load file",
    condition=test_load_txt(
        path=data_path
    )
)

✅ PASSED: load_txt: load file.


### Data type

In [6]:
# === Input ===
# Pick a file to load and the data type
data_path = f"{fake_input_dir}/fake_data_float.dat"
dtype = float

# === Test & Record ===
record_test(
    test_name="load_txt: dtype",
    condition=test_load_txt(
        path=data_path,
        dtype=dtype
    )
)

✅ PASSED: load_txt: dtype.


### Skiprows

In [7]:
# === Input ===
# Pick a file to load and the rows to skip
data_path = f"{fake_input_dir}/fake_data_float.dat"
skiprows = 2

# === Test & Record ===
record_test(
    test_name="load_txt: skiprows",
    condition=test_load_txt(
        path=data_path,
        skiprows=skiprows
    )
)

✅ PASSED: load_txt: skiprows.


### Usecols

In [8]:
# === Input ===
# Pick a file to load and the colums to read
data_path = f"{fake_input_dir}/fake_data_float.dat"
usecols = (0, 1)

# === Test & Record ===
record_test(
    test_name="load_txt: usecols",
    condition=test_load_txt(
        path=data_path,
        usecols=usecols
    )
)

✅ PASSED: load_txt: usecols.


### Start and stop

In [9]:
# === Input ===
# Pick a file to load and the slice indeces
data_path = f"{fake_input_dir}/fake_data_float.dat"
start = 1
stop = 3

# === Test & Record ===
record_test(
    test_name="load_txt: start",
    condition=test_load_txt(
        path=data_path,
        start=start
    )
)

# === Test & Record ===
record_test(
    test_name="load_txt: stop",
    condition=test_load_txt(
        path=data_path,
        stop=stop
    )
)

# === Test & Record ===
record_test(
    test_name="load_txt: start and stop",
    condition=test_load_txt(
        path=data_path,
        start=start,
        stop=stop
    )
)

✅ PASSED: load_txt: start.
✅ PASSED: load_txt: stop.
✅ PASSED: load_txt: start and stop.


### Skiprows w/ start and stop

In [10]:
# === Input ===
# Pick a file to load, the rows to skip and the slice indeces
data_path = f"{fake_input_dir}/fake_data_float.dat"
skiprows = 0
start = 1
stop = 3

# === Test & Record ===
record_test(
    test_name="load_txt: skiprows with start and stop",
    condition=test_load_txt(
        path=data_path,
        skiprows=skiprows,
        start=start,
        stop=stop
    )
)

✅ PASSED: load_txt: skiprows with start and stop.


### Invalid file path

In [11]:
# === Input ===
# Pick a non-existing file to load and the expected error
data_path = f"{fake_input_dir}/wrong_name.dat"
expected_error = FileNotFoundError

# === Test & Record ===
record_test(
    test_name="load_txt: invalid file path",
    condition=test_raises(
        func=lambda: jkio.load_txt(
            path=data_path,
        ),
        expected_error=expected_error,
    )
)

✅ PASSED: load_txt: invalid file path.


### Invalid data type

In [12]:
# === Input ===
# Pick a non-existing file to load, a wrong type
# and the expected error
data_path = f"{fake_input_dir}/fake_data_float.dat"
dtype = int
expected_error = ValueError

# === Test & Record ===
record_test(
    test_name="load_txt: invalid data type",
    condition=test_raises(
        func=lambda: jkio.load_txt(
            path=data_path,
            dtype=dtype
        ),
        expected_error=expected_error,
    )
)

✅ PASSED: load_txt: invalid data type.


### Invalid usecols

In [13]:
# === Input ===
# Pick a non-existing file to load, wrong colums
# and the expected error
data_path = f"{fake_input_dir}/fake_data_float.dat"
usecols = 32
expected_error = ValueError

# === Test & Record ===
record_test(
    test_name="load_txt: invalid columns",
    condition=test_raises(
        func=lambda: jkio.load_txt(
            path=data_path,
            usecols=usecols
        ),
        expected_error=expected_error,
    )
)

✅ PASSED: load_txt: invalid columns.


## Test 2 - Data saving

In [14]:
def test_save_txt(
    path,
    data,
    dfmt = "%.18e",
    header = "",
    transpose = False
):

    data = np.asarray(data)

    # Save via numpy
    path_numpy = f"{path}_numpy.dat"

    if transpose:
        temp_data = data.T
    else:
        temp_data = data

    np.savetxt(
        fname=path_numpy,
        X=temp_data,
        fmt=dfmt,
        header=header,
    )

    # Save via jklab
    path = f"{path}.dat"

    jkio.save_txt(
        path=path,
        data=data,
        dfmt=dfmt,
        header=header,
        transpose=transpose
    )

    # Load and compare
    with open(path_numpy, "r") as file:
        numpy_data = file.read()

    with open(path, "r") as file:
        jklab_data = file.read()

    test = (numpy_data == jklab_data)

    return test


def test_save_txt_error(
    path,
    data,
    dfmt="%.18e",
    header="",
    transpose=False,
    expected_error=ValueError,
):

    try:

        jkio.save_txt(
            path=path,
            data=data,
            dfmt=dfmt,
            header=header,
            transpose=transpose
        )

    except expected_error:

        return True

    except Exception:

        return False

    return False

def test_save_txt_warning(
    path,
    data,
    dfmt,
    expected_warning=UserWarning,
    expected_message="",
):

    path = f"{path}.dat"

    with pytest.warns(expected_warning) as warning_record:

        jkio.save_txt(
            path=path,
            data=data,
            dfmt=dfmt,
        )

    # Check that exactly one warning was issued
    warning_count_match = (
        len(warning_record)
        == 1
    )

    # Check warning category
    warning_type_match = (
        warning_record[0].category
        is expected_warning
    )

    # Check warning message
    warning_message_match = (
        str(warning_record[0].message)
        == expected_message
    )

    test = (
        warning_count_match
        and warning_type_match
        and warning_message_match
    )

    return test

### Basic saving 

In [15]:
# === Input ===
# Pick a file to save and its content
savefile_path = f"{fake_output_dir}/fake_data"
data_to_save = np.array([
    [1.1, 1.2, 1.3],
    [2.1, 2.2, 2.3],
    [3.1, 3.2, 3.3],
    [4.1, 4.2, 4.3],
])

# === Test & Record ===
record_test(
    test_name="save_txt: save file",
    condition=test_save_txt(
        path=savefile_path,
        data=data_to_save
    )
)

✅ PASSED: save_txt: save file.


### Output formatting

In [16]:
# === Input ===
# Pick a file to save, its content and a format
savefile_path = f"{fake_output_dir}/fake_data"
data_to_save = np.array([
    [1.1, 1.2, 1.3],
    [2.1, 2.2, 2.3],
    [3.1, 3.2, 3.3],
    [4.1, 4.2, 4.3],
])
dmft = "%d"

# === Test & Record ===
record_test(
    test_name="save_txt: save file with format",
    condition=test_save_txt(
        path=savefile_path,
        data=data_to_save,
        dfmt=dmft
    )
)

✅ PASSED: save_txt: save file with format.


/Users/menny/Code/myPython/JKateLab/jklab-core/src/jklab/core/io.py:188: UserWarning: Integer formatting may cause loss of numerical information.

Context:
data.dtype = dtype('float64')
dfmt = '%d'
  jkex.raise_warning(


### File header

In [17]:
# === Input ===
# Pick a file to save, its content and an header
savefile_path = f"{fake_output_dir}/fake_data"
data_to_save = np.array([
    [1.1, 1.2, 1.3],
    [2.1, 2.2, 2.3],
    [3.1, 3.2, 3.3],
    [4.1, 4.2, 4.3],
])
header = "fake header"

# === Test & Record ===
record_test(
    test_name="save_txt: save file with header",
    condition=test_save_txt(
        path=savefile_path,
        data=data_to_save,
        header=header
    )
)

✅ PASSED: save_txt: save file with header.


### Transposed data

In [18]:
# === Input ===
# Pick a file to save, its content and 
# the transpose flag
savefile_path = f"{fake_output_dir}/fake_data"
data_to_save = np.array([
    [1.1, 1.2, 1.3],
    [2.1, 2.2, 2.3],
    [3.1, 3.2, 3.3],
    [4.1, 4.2, 4.3],
])
transpose = True

# === Test & Record ===
record_test(
    test_name="save_txt: save file with transposed data",
    condition=test_save_txt(
        path=savefile_path,
        data=data_to_save,
        transpose=transpose
    )
)

✅ PASSED: save_txt: save file with transposed data.


### Wrong Transposed data

In [19]:
# === Input ===
# Pick a file to save, its (non-2d) content  
# the transpose flag and the expected error
savefile_path = f"{fake_output_dir}/fake_data"
data_to_save1d = np.array([
    1.1, 1.2, 1.3
])
data_to_save3d = np.array([
    [
        [1.1, 1.2, 1.3],
        [2.1, 2.2, 2.3],
        [3.1, 3.2, 3.3],
        [4.1, 4.2, 4.3],
    ],
    [
        [1.1, 1.2, 1.3],
        [2.1, 2.2, 2.3],
        [3.1, 3.2, 3.3],
        [4.1, 4.2, 4.3],
    ]
])
transpose = True
error = ValueError

# === Test & Record ===
record_test(
    test_name="save_txt: save file with transpose error",
    condition=test_save_txt_error(
        path=savefile_path,
        data=data_to_save3d,
        transpose=transpose,
        expected_error=error
    )
)

✅ PASSED: save_txt: save file with transpose error.


### Check format: float data and int fmt

In [20]:
# # === Input ===
# # Pick a file to save, its content and a format
# # NOTE: warning message was copied from io.py module
savefile_path = f"{fake_output_dir}/fake_data"
data_to_save = np.array([
    [1.1, 1.2, 1.3],
    [2.1, 2.2, 2.3],
    [3.1, 3.2, 3.3],
    [4.1, 4.2, 4.3],
])
dfmt = "%d"
expected_warning = UserWarning
expected_message = jkex.format_message(
    message=("Integer formatting may cause loss of "
             "numerical information."),
    context={
        "data.dtype": data_to_save.dtype,
        "dfmt": dfmt
    }
)

# # === Test & Record ===
record_test(
    test_name="save_txt: float data and int format",
    condition=test_save_txt_warning(
        path=savefile_path,
        data=data_to_save,
        dfmt=dfmt,
        expected_warning=expected_warning,
        expected_message=expected_message
    )
)

✅ PASSED: save_txt: float data and int format.


### Check format: float data and float fmt

In [21]:
# # === Input ===
# # Pick a file to save, its content and a format
# # NOTE: warning message was copied from io.py module
savefile_path = f"{fake_output_dir}/fake_data"
data_to_save = np.array([
    [1.1, 1.2, 1.3],
    [2.1, 2.2, 2.3],
    [3.1, 3.2, 3.3],
    [4.1, 4.2, 4.3],
])
dfmt = "%f"
expected_warning = UserWarning
expected_message = jkex.format_message(
    message=("Integer formatting may cause loss of "
             "numerical information."),
    context={
        "data.dtype": data_to_save.dtype,
        "dfmt": dfmt
    }
)

# # === Test & Record ===
record_test(
    test_name="save_txt: float data and float format",
    condition=test_save_txt(
        path=savefile_path,
        data=data_to_save,
        dfmt=dfmt,
    )
)

✅ PASSED: save_txt: float data and float format.


### Check format: int data and int fmt

In [22]:
# # === Input ===
# # Pick a file to save, its content and a format
# # NOTE: warning message was copied from io.py module
savefile_path = f"{fake_output_dir}/fake_data"
data_to_save_int = np.array([
    [1, 1, 1],
    [2, 2, 2],
    [3, 3, 3],
    [4, 4, 4],
])
dfmt = "%d"
expected_warning = UserWarning
expected_message = jkex.format_message(
    message=("Integer formatting may cause loss of "
             "numerical information."),
    context={
        "data.dtype": data_to_save.dtype,
        "dfmt": dfmt
    }
)

# # === Test & Record ===
record_test(
    test_name="save_txt: int data and int format",
    condition=test_save_txt(
        path=savefile_path,
        data=data_to_save_int,
        dfmt=dfmt,
    )
)

✅ PASSED: save_txt: int data and int format.


### Check format: int data and float fmt

In [23]:
# # === Input ===
# # Pick a file to save, its content and a format
# # NOTE: warning message was copied from io.py module
savefile_path = f"{fake_output_dir}/fake_data"
data_to_save_int = np.array([
    [1, 1, 1],
    [2, 2, 2],
    [3, 3, 3],
    [4, 4, 4],
])
dfmt = "%f"
expected_warning = UserWarning
expected_message = jkex.format_message(
    message=("Integer formatting may cause loss of "
             "numerical information."),
    context={
        "data.dtype": data_to_save.dtype,
        "dfmt": dfmt
    }
)

# # === Test & Record ===
record_test(
    test_name="save_txt: int data and float format",
    condition=test_save_txt(
        path=savefile_path,
        data=data_to_save_int,
        dfmt=dfmt,
    )
)

✅ PASSED: save_txt: int data and float format.


# Summary

In [24]:
print_test_summary(test_results)


TEST SUMMARY
Passed: 20/20
Failed: 0/20
Success rate: 100.0%

PASSED: load_txt: load file
PASSED: load_txt: dtype
PASSED: load_txt: skiprows
PASSED: load_txt: usecols
PASSED: load_txt: start
PASSED: load_txt: stop
PASSED: load_txt: start and stop
PASSED: load_txt: skiprows with start and stop
PASSED: load_txt: invalid file path
PASSED: load_txt: invalid data type
PASSED: load_txt: invalid columns
PASSED: save_txt: save file
PASSED: save_txt: save file with format
PASSED: save_txt: save file with header
PASSED: save_txt: save file with transposed data
PASSED: save_txt: save file with transpose error
PASSED: save_txt: float data and int format
PASSED: save_txt: float data and float format
PASSED: save_txt: int data and int format
PASSED: save_txt: int data and float format
